In [1]:
from splinter import Browser
from bs4 import BeautifulSoup as soup 
import re
import pandas as pd
import numpy as np
import time
import json
import random

In [2]:
browser = Browser('chrome')
city = "Mumbai"
target_cars = 4500
cars_collected = 0
total_pages = 250 

In [ ]:
def collect_car_links(city, total_pages, target_cars):

    cars_collected = 0

    with open(f"car_links_{city}.txt", "a") as f:

        for page_num in range(1, total_pages + 1):

            if page_num == 1:
                url = f"https://www.cardekho.com/used-cars+in+{city}"
            else:
                url = f"https://www.cardekho.com/used-cars+in+{city}/page-{page_num}"

            browser.visit(url)
            time.sleep(2)

            browser.execute_script("window.scrollTo(0, 1000);")
            time.sleep(1)

            current_soup = soup(browser.html, 'html.parser')

            page_links_found = 0

            for link in current_soup.find_all('a', href=True):
                href = link['href']

                if 'used-car-details' in href:

                    full_url = f"https://www.cardekho.com{href}" if href.startswith('/') else href

                    f.write(full_url + "\n")

                    cars_collected += 1
                    page_links_found += 1

            print(f"Page {page_num}: Saved {page_links_found} links. Total: {cars_collected}")

            if cars_collected >= target_cars:
                print("Target reached!")
                break

    print(f"All links are now saved in car_links_{city.lower()}.txt")

In [5]:
collect_car_links(city, total_pages, target_cars)

Page 1: Saved 28 links. Total: 28
Page 2: Saved 28 links. Total: 56
Page 3: Saved 28 links. Total: 84
Page 4: Saved 28 links. Total: 112
Page 5: Saved 28 links. Total: 140
Page 6: Saved 20 links. Total: 160
Page 7: Saved 20 links. Total: 180
Page 8: Saved 20 links. Total: 200
Page 9: Saved 20 links. Total: 220
Page 10: Saved 20 links. Total: 240
Page 11: Saved 20 links. Total: 260
Page 12: Saved 20 links. Total: 280
Page 13: Saved 20 links. Total: 300
Page 14: Saved 20 links. Total: 320
Page 15: Saved 20 links. Total: 340
Page 16: Saved 20 links. Total: 360
Page 17: Saved 20 links. Total: 380
Page 18: Saved 20 links. Total: 400
Page 19: Saved 20 links. Total: 420
Page 20: Saved 20 links. Total: 440
Page 21: Saved 20 links. Total: 460
Page 22: Saved 20 links. Total: 480
Page 23: Saved 20 links. Total: 500
Page 24: Saved 20 links. Total: 520
Page 25: Saved 20 links. Total: 540
Page 26: Saved 20 links. Total: 560
Page 27: Saved 20 links. Total: 580
Page 28: Saved 20 links. Total: 600
Page

## Main Extraction 

In [ ]:
def scrape_car_links(city, links_filename="car_links.txt"):

    def get_browser():
        return Browser('chrome')

    # 1. Load links
    with open(links_filename, "r") as f:
        all_links = [line.strip() for line in f.readlines()]

    output_file = f"car_dataset_{city.lower()}.json"
    browser = get_browser()

    for index, link in enumerate(all_links):
        try:
            print(f"Scraping {index+1}/{len(all_links)}: {link}")

            # Visit page
            browser.visit(link)

            # Human delay + scroll
            time.sleep(random.uniform(1, 2))
            browser.execute_script("window.scrollTo(0, 600);")
            time.sleep(0.5)

            # Expand specifications
            try:
                view_all_spec_btn = browser.find_by_text('View all Specifications')
                if view_all_spec_btn:
                    browser.execute_script(
                        "arguments[0].click();",
                        view_all_spec_btn.first._element
                    )
                    print("Expanded specifications.")
                    time.sleep(0.6)
            except Exception:
                pass

            # Parse page
            page_soup = soup(browser.html, 'html.parser')

            car_data = {"url": link}

            # -------------------------
            # CAR NAME EXTRACTION
            # -------------------------
            name_tag = page_soup.find('div', class_='vehicleName')
            h1 = name_tag.find('h1') if (name_tag and name_tag.find('h1')) else page_soup.find('h1')

            if h1:
                parts = h1.get_text(separator="|", strip=True).split("|")
                car_data["car_name"] = parts[1].strip() if len(parts) >= 2 else parts[0].strip()

            # -------------------------
            # PRICE EXTRACTION
            # -------------------------
            price_div = page_soup.find('div', class_='vehiclePrice')
            if price_div:
                price_span = price_div.find('span')
                if price_span:
                    car_data["Price"] = price_span.get_text(strip=True)

            # -------------------------
            # SPECIFICATIONS EXTRACTION
            # -------------------------
            spec_items = page_soup.find_all('li', class_='gsc_col-xs-12')

            for item in spec_items:
                label_tag = item.find('div', class_='label')
                value_tag = item.find('span', class_='value-text')

                if label_tag and value_tag:
                    label = label_tag.get_text(strip=True)
                    value = value_tag.get_text(strip=True)
                    car_data[label] = value

            # Save data
            if len(car_data) > 1:
                with open(output_file, "a") as out:
                    out.write(json.dumps(car_data) + "\n")

                print(f"Saved: {car_data.get('Price','N/A')} and {len(car_data)-2} specs.")
            else:
                print(f"No data found for: {link}")

        except Exception as e:
            print(f"Serious error at {link}: {e}")

            browser.quit()
            browser = get_browser()
            time.sleep(1)
            continue

    browser.quit()

In [7]:
scrape_car_links(f"{city}", f"car_links_{city}.txt")

Scraping 1/4500: https://www.cardekho.com/used-car-details/used-Honda-amaze-e-petrol-cars-Mumbai_b626b26c-cb97-44a5-aea5-af134fee3f20.htm?adId=23204&adType=41
Expanded specifications.
Saved: ₹5.40 Lakh and 16 specs.
Scraping 2/4500: https://www.cardekho.com/buy-used-car-details/used-Maruti-vitara-brezza-vxi-cars-Mumbai_11f77123-a37b-4adf-918c-2fa5a2d844c4.htm
Expanded specifications.
Saved: ₹6.49 Lakh and 44 specs.
Scraping 3/4500: https://www.cardekho.com/used-car-details/used-Skoda-kushaq-15-tsi-style-bsvi-cars-Thane_c6c96884-2ebf-474b-982a-8a3ed11fec11.htm
Expanded specifications.
Saved: ₹9.45 Lakh and 50 specs.
Scraping 4/4500: https://www.cardekho.com/used-car-details/used-Volkswagen-ameo-12-mpi-comfortline-cars-Mumbai_0dab4260-e443-47f1-967f-469edd2adafe.htm
Expanded specifications.
Saved: ₹3.55 Lakh and 48 specs.
Scraping 5/4500: https://www.cardekho.com/used-car-details/used-Maruti-ciaz-alpha-automatic-bsiv-cars-Mumbai_66e94287-ac2a-4ea3-9bb8-266e20845197.htm?adId=22831&adType=

There was an error managing chromedriver (error sending request for url (https://googlechromelabs.github.io/chrome-for-testing/known-good-versions-with-downloads.json)); using driver found in the cache
Error sending stats to Plausible: error sending request for url (https://plausible.io/api/event)


Scraping 1596/4500: https://www.cardekho.com/used-car-details/used-Mg-hector-plus-15-turbo-sharp-pro-cvt-bsvi-cars-Mumbai_a9b9614f-22e8-4453-8704-625e97a0a2db.htm
Saved: N/A and 0 specs.
Scraping 1597/4500: https://www.cardekho.com/used-car-details/used-Kia-seltos-gtx-plus-s-turbo-dct-cars-Mumbai_fb0fbbf8-6038-4823-9be6-2c78922a7c11.htm
Saved: N/A and 0 specs.
Scraping 1598/4500: https://www.cardekho.com/used-car-details/used-Toyota-hyryder-v-hybrid-cars-Mumbai_eff70606-3144-4196-a9da-94aacfc6f5a2.htm
Serious error at https://www.cardekho.com/used-car-details/used-Toyota-hyryder-v-hybrid-cars-Mumbai_eff70606-3144-4196-a9da-94aacfc6f5a2.htm: Message: unknown error: net::ERR_INTERNET_DISCONNECTED
  (Session info: chrome=146.0.7680.165)
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff7770a29c5+2ed785]
	chromedriver!GetHandleVerifier [0x7ff776dca0d0+14e90]
	chromedriver!(No symbol) [0x7ff776b2db2d]
	chromedriver!(No symbol) [0x7ff776b2a654]
	chromedriver!(No symbol) [0x7ff776b1a6d1]
	chr

There was an error managing chromedriver (error sending request for url (https://googlechromelabs.github.io/chrome-for-testing/known-good-versions-with-downloads.json)); using driver found in the cache


Scraping 1599/4500: https://www.cardekho.com/used-car-details/used-Mercedes-benz-c-class-c-220-cdi-style-cars-Mumbai_a8fc88ff-1870-4e3c-8d53-34249822202f.htm
Serious error at https://www.cardekho.com/used-car-details/used-Mercedes-benz-c-class-c-220-cdi-style-cars-Mumbai_a8fc88ff-1870-4e3c-8d53-34249822202f.htm: Message: unknown error: net::ERR_INTERNET_DISCONNECTED
  (Session info: chrome=146.0.7680.165)
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff7770a29c5+2ed785]
	chromedriver!GetHandleVerifier [0x7ff776dca0d0+14e90]
	chromedriver!(No symbol) [0x7ff776b2db2d]
	chromedriver!(No symbol) [0x7ff776b2a654]
	chromedriver!(No symbol) [0x7ff776b1a6d1]
	chromedriver!(No symbol) [0x7ff776b1c5bf]
	chromedriver!(No symbol) [0x7ff776b1ac73]
	chromedriver!(No symbol) [0x7ff776b1a43b]
	chromedriver!(No symbol) [0x7ff776b1a0f1]
	chromedriver!(No symbol) [0x7ff776b17d3b]
	chromedriver!(No symbol) [0x7ff776b18482]
	chromedriver!(No symbol) [0x7ff776b31de6]
	chromedriver!(No symbol) [0x7ff776bd4

...

Scraping 1602/4500: https://www.cardekho.com/used-car-details/used-Toyota-hyryder-v-hybrid-cars-Mumbai_0d352545-eecf-42c8-bd15-779ffd3ce399.htm
Saved: N/A and 0 specs.
Scraping 1603/4500: https://www.cardekho.com/used-car-details/used-Kia-seltos-htx-diesel-at-cars-Mumbai_27fa6298-6890-4d22-b7c5-564de50ba5dd.htm
Saved: N/A and 0 specs.
Scraping 1604/4500: https://www.cardekho.com/used-car-details/used-Toyota-hyryder-v-hybrid-cars-Mumbai_eff70606-3144-4196-a9da-94aacfc6f5a2.htm
Saved: N/A and 0 specs.
Scraping 1605/4500: https://www.cardekho.com/used-car-details/used-Mg-hector-sharp-cvt-cars-Mumbai_a8ddd4fa-4440-44c5-b7f5-0cecb74d6c27.htm
Serious error at https://www.cardekho.com/used-car-details/used-Mg-hector-sharp-cvt-cars-Mumbai_a8ddd4fa-4440-44c5-b7f5-0cecb74d6c27.htm: Message: unknown error: net::ERR_INTERNET_DISCONNECTED
  (Session info: chrome=146.0.7680.165)
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff7770a29c5+2ed785]
	chromedriver!GetHandleVerifier [0x7ff776dca0d0+14e90]

There was an error managing chromedriver (error sending request for url (https://googlechromelabs.github.io/chrome-for-testing/known-good-versions-with-downloads.json)); using driver found in the cache


Scraping 1606/4500: https://www.cardekho.com/buy-used-car-details/used-Maruti-baleno-alpha-diesel-cars-Mumbai_bff8f0a5-7fe4-4009-8fd0-6658a61a1aac.htm
Serious error at https://www.cardekho.com/buy-used-car-details/used-Maruti-baleno-alpha-diesel-cars-Mumbai_bff8f0a5-7fe4-4009-8fd0-6658a61a1aac.htm: Message: unknown error: net::ERR_INTERNET_DISCONNECTED
  (Session info: chrome=146.0.7680.165)
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff7770a29c5+2ed785]
	chromedriver!GetHandleVerifier [0x7ff776dca0d0+14e90]
	chromedriver!(No symbol) [0x7ff776b2db2d]
	chromedriver!(No symbol) [0x7ff776b2a654]
	chromedriver!(No symbol) [0x7ff776b1a6d1]
	chromedriver!(No symbol) [0x7ff776b1c5bf]
	chromedriver!(No symbol) [0x7ff776b1ac73]
	chromedriver!(No symbol) [0x7ff776b1a43b]
	chromedriver!(No symbol) [0x7ff776b1a0f1]
	chromedriver!(No symbol) [0x7ff776b17d3b]
	chromedriver!(No symbol) [0x7ff776b18482]
	chromedriver!(No symbol) [0x7ff776b31de6]
	chromedriver!(No symbol) [0x7ff776bd4994]
	chromedr

There was an error managing chromedriver (error sending request for url (https://googlechromelabs.github.io/chrome-for-testing/known-good-versions-with-downloads.json)); using driver found in the cache


Scraping 1607/4500: https://www.cardekho.com/buy-used-car-details/used-Honda-mobilio-v-i-vtec-cars-Mumbai_fc0b6a43-04b5-40be-94ff-537978551050.htm
Expanded specifications.
Saved: ₹4.22 Lakh and 52 specs.
Scraping 1608/4500: https://www.cardekho.com/buy-used-car-details/used-Renault-triber-rxz-cars-Mumbai_e0bbdb1e-b379-4511-84e4-d46c5e697e67.htm
Expanded specifications.
Saved: ₹3.61 Lakh and 47 specs.
Scraping 1609/4500: https://www.cardekho.com/buy-used-car-details/used-Maruti-baleno-12-alpha-cars-Mumbai_c7129c1f-4875-4296-ac79-bb75f60ae8f9.htm
Expanded specifications.
Saved: ₹4.68 Lakh and 52 specs.
Scraping 1610/4500: https://www.cardekho.com/buy-used-car-details/used-Maruti-baleno-12-delta-cars-Mumbai_42309f6e-3f97-4396-b39a-199d13a1821e.htm
Expanded specifications.
Saved: ₹4.05 Lakh and 54 specs.
Scraping 1611/4500: https://www.cardekho.com/buy-used-car-details/used-Hyundai-xcent-12-vtvt-s-at-cars-Mumbai_f8a79259-4d5b-403e-8400-8800346b75d7.htm
Expanded specifications.
Saved: ₹3.80